# Getting Started with mycontext-ai

Welcome to **mycontext-ai** - The Universal Context Transformation Engine!

This notebook will guide you through:
1. Installation
2. Basic Context Creation
3. Using Cognitive Templates
4. Working with LLM Providers
5. Advanced Features (RAG, Sessions, Structured Outputs)

## 1. Installation

First, let's install mycontext-ai from PyPI:

In [ ]:
# Install the base package
#!pip install mycontext-ai

# Or install with LLM execution support (uses LiteLLM for all providers)
!pip install mycontext-ai litellm

## 2. Basic Context Creation

The core of mycontext-ai is the `Context` class - a structured way to transform questions into perfect contexts.

In [ ]:
from mycontext import Context, Directive, Guidance

# Create a simple context
context = Context(
    directive=Directive(
        content="Help debug this Python code and identify issues",
        priority=8,
        constraints=["Be concise", "Provide examples"]
    ),
    guidance=Guidance(
        role="Experienced Senior Python Developer",
        rules=["Always explain your reasoning", "Use best practices"],
        style="Clear and educational"
    )
)

# View the assembled context
print(context.assemble())

### Export Context to Different Formats

In [ ]:
# As dictionary (for APIs)
print("\n=== Dictionary Format ===")
print(context.to_dict())

# As LangChain messages
print("\n=== LangChain Format ===")
print(context.to_langchain())

# As Markdown
print("\n=== Markdown Format ===")
print(context.to_markdown())

## 3. Using Cognitive Templates

mycontext-ai includes research-backed cognitive templates for better thinking.

### Question Analyzer Template

In [ ]:
from mycontext.templates.free import QuestionAnalyzer

# Create the analyzer template
analyzer = QuestionAnalyzer()

# Build the context with your question
analysis_context = analyzer.build_context(
    question="How can I improve my Python code's performance?",
    depth="comprehensive",
    context="The code processes large CSV files"
)

print(analysis_context.assemble())

### Step-by-Step Reasoner Template

In [ ]:
from mycontext.templates.free import StepByStepReasoner

# Create the reasoner template
reasoner = StepByStepReasoner()

# Build the context with your problem
reasoning_context = reasoner.build_context(
    problem="Design a scalable microservices architecture",
    goal="Handle 1M requests per day",
    constraints=["Budget: $1000/month", "Use cloud services"],
    context="E-commerce platform with payment processing"
)

print(reasoning_context.assemble())

### Socratic Questioner Template

In [ ]:
from mycontext.templates.free import SocraticQuestioner

# Create the Socratic questioner template
questioner = SocraticQuestioner()

# Build the context with your statement
socratic_context = questioner.build_context(
    statement="We should use microservices for everything",
    context="Team of 5 developers, startup environment"
)

print(socratic_context.assemble())

### Comparative Analyzer Template

In [ ]:
from mycontext.templates.free import ComparativeAnalyzer

# Create the comparative analyzer template
analyzer = ComparativeAnalyzer()

# Build the context with your comparison
# Note: options can be a list, but criteria should be comma-separated string
comparison_context = analyzer.build_context(
    options=["PostgreSQL", "MongoDB", "Redis"],
    criteria="Performance, Scalability, Cost, Learning curve",
    context="Building a real-time chat application"
)

print(comparison_context.assemble())

### Risk Assessor Template

In [ ]:
from mycontext.templates.free import RiskAssessor

# Create the risk assessor template
assessor = RiskAssessor()

# Build the context with your risk assessment
risk_context = assessor.build_context(
    decision="Migrate to Kubernetes",
    scope="Production infrastructure",
    context="Currently using traditional VMs, 50 services"
)

print(risk_context.assemble())

## 4. Working with LLM Providers

mycontext-ai works seamlessly with OpenAI, Anthropic, and Google Gemini.

### Setup API Keys

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = 'your-openai-api-key-here'
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
os.environ['GOOGLE_API_KEY'] = 'your-google-api-key-here'

### Using OpenAI

In [ ]:
# LLM execution requires litellm: pip install litellm
# Provider-specific SDK (optional): pip install openai

from mycontext.templates.free import QuestionAnalyzer

# Create a template
analyzer = QuestionAnalyzer()

# Execute with OpenAI (pass provider as string)
result = analyzer.execute(
    provider="openai",  # Provider name as string, not instance!
    question="What are the best practices for API design?",
    depth="comprehensive",
    model="gpt-4o-mini",
    temperature=0.7
)

print(f"Response: {result.response}")
print(f"\nTokens used: {result.tokens_used}")
print(f"Cost: ${result.cost_usd:.4f}")

### Using Anthropic Claude

In [ ]:
# LLM execution requires litellm: pip install litellm
# Provider-specific SDK (optional): pip install anthropic

from mycontext.templates.free import StepByStepReasoner

# Create a template
reasoner = StepByStepReasoner()

# Execute with Claude (pass provider as string)
result = reasoner.execute(
    provider="anthropic",  # Provider name as string!
    problem="Optimize database queries",
    goal="Reduce query time by 50%",
    model="claude-3-5-sonnet-20241022",
    temperature=0.7
)

print(f"Response: {result.response}")
print(f"\nTokens used: {result.tokens_used}")
print(f"Cost: ${result.cost:.4f}")

### Using Google Gemini

In [ ]:
# LLM execution requires litellm: pip install litellm
# Provider-specific SDK (optional): pip install google-genai

from mycontext.templates.free import ComparativeAnalyzer

# Create a template
analyzer = ComparativeAnalyzer()

# Execute with Gemini (pass provider as string)
# Note: criteria should be comma-separated string
result = analyzer.execute(
    provider="gemini",  # Provider name as string!
    options=["REST API", "GraphQL", "gRPC"],
    criteria="Performance, Developer Experience, Ecosystem",
    model="gemini-2.5-flash",
    temperature=0.7
)

print(f"Response: {result.response}")
print(f"\nTokens used: {result.tokens_used}")

## 5. Advanced Features

### Structured Outputs

In [ ]:
from pydantic import BaseModel

from mycontext import Context, Directive, Guidance
from mycontext.utils.structured_output import output_format


# Define your schema
class APIDesign(BaseModel):
    endpoints: list[str]
    methods: list[str]
    authentication: str

# Create schema dict for output_format function
schema_dict = {
    "endpoints": "list of strings",
    "methods": "list of strings",
    "authentication": "string"
}

# Create context with structured output
context = Context(
    guidance=Guidance(
        role="API Designer"
    ),
    directive=Directive(
        content=f"Design a RESTful API for a blog platform\n\n{output_format('json', schema=schema_dict)}"
    )
)

print(context.assemble())

### Session Management (Multi-turn Conversations)

In [ ]:
from mycontext.knowledge.session import Session

# Create a session with memory
session = Session(
    session_id="code-review-session",
    token_budget=4000,
    system_prompt="You are a code reviewer focused on Python best practices."
)

# Turn 1
response1 = session.send(
    "What are the main issues with this code: def calc(x,y): return x+y",
    provider="openai",  # Provider name as string!
    model="gpt-4o-mini"
)
print(f"Turn 1: {response1.response}\n")

# Turn 2 - Session remembers context
response2 = session.send(
    "Can you show me the corrected version?",
    provider="openai",
    model="gpt-4o-mini"
)
print(f"Turn 2: {response2.response}\n")

# View session history
print(f"Total turns: {len(session.history)}")
print(f"Token usage: {session.current_tokens}/{session.token_budget}")

### RAG (Retrieval-Augmented Generation)

In [ ]:
# Install RAG features
# !pip install mycontext-ai[rag]

from mycontext.knowledge.rag.chunking import SemanticChunker
from mycontext.knowledge.rag.embeddings import OpenAIEmbedding
from mycontext.knowledge.rag.retriever import Retriever
from mycontext.knowledge.rag.vector_store import InMemoryVectorStore

# Sample documents
documents = [
    "Python is a high-level programming language known for readability.",
    "FastAPI is a modern web framework for building APIs with Python.",
    "Type hints in Python improve code quality and IDE support.",
    "Async/await in Python enables efficient concurrent programming."
]

# Create RAG pipeline
chunker = SemanticChunker(chunk_size=100)
embedder = OpenAIEmbedding()
vector_store = InMemoryVectorStore(embedding_dim=1536)

# Process and store documents
for doc in documents:
    chunks = chunker.chunk(doc)
    for chunk in chunks:
        embedding = embedder.embed(chunk.content)
        vector_store.add(chunk.content, embedding)

# Create retriever
retriever = Retriever(vector_store=vector_store, embedder=embedder)

# Search for relevant context
query = "How do I build web APIs in Python?"
results = retriever.retrieve(query, top_k=2)

print(f"Query: {query}\n")
for i, result in enumerate(results, 1):
    print(f"Result {i} (score: {result['score']:.4f}):")
    print(f"{result['content']}\n")

# Use with Context
from mycontext import Context, Directive, Guidance

context = Context(
    guidance=Guidance(
        role="Python Expert"
    ),
    directive=Directive(
        content="Answer the question based on the provided knowledge"
    ),
    knowledge="\n".join([r['content'] for r in results])
)

print("\n=== Context with RAG ===")
print(context.assemble())

### Blueprints (Complex Context Architectures)

In [ ]:
from mycontext.structure.blueprint import Blueprint

# Create a complex context blueprint
blueprint = Blueprint(
    name="code-review-blueprint",
    description="Comprehensive code review context",
    components=[
        "Review for security vulnerabilities",
        "Check code style and formatting",
        "Assess performance implications",
        "Verify test coverage"
    ],
    token_budget=2000
)

# Build the context
context = blueprint.build()
print(context.assemble())

# Optimize for different strategies
fast_context = blueprint.optimize("speed")
quality_context = blueprint.optimize("quality")
cheap_context = blueprint.optimize("cost")

print(f"\nFast context tokens: ~{blueprint.estimate_tokens()}")

## 6. Universal Portability

Export contexts to any format for use with other frameworks:

In [ ]:
from mycontext.templates.free import QuestionAnalyzer

# Create a rich context
analyzer = QuestionAnalyzer()
context = analyzer.build_context(
    question="How does transformer architecture work?",
    depth="comprehensive",
    context="Explain for ML engineers"
)

# Export for different frameworks
print("=== For LangChain ===")
print(context.to_langchain())

print("\n=== For LlamaIndex ===")
print(context.to_dict())

print("\n=== For API Calls ===")
print(context.to_messages())

print("\n=== For Documentation ===")
print(context.to_markdown())

print("\n=== For Storage ===")
print(context.to_json())

## 📚 Real-World Example: Research a Topic and Create a Guide

Let's say you want to research **Terraform** and create a beginner's guide in markdown format.
Here's the complete workflow:

In [ ]:
# EXAMPLE: Research Terraform and create a beginner's guide

from mycontext import Context, Directive, Guidance
from mycontext.templates.free import QuestionAnalyzer, StepByStepReasoner

# Configuration
TOPIC = "Terraform"

# ============================================================================
# STEP 1: Understand the fundamentals
# ============================================================================
analyzer = QuestionAnalyzer()
fundamentals = analyzer.execute(
    provider="openai",
    question=f"What are the core concepts a beginner should understand about {TOPIC}?",
    depth="comprehensive",
    context="Complete beginner to Infrastructure as Code",
    model="gpt-4o-mini"
)

print("FUNDAMENTALS:")
print(fundamentals.response[:500] + "...\n")  # Preview

# ============================================================================
# STEP 2: Create a practical getting started guide
# ============================================================================
reasoner = StepByStepReasoner()
tutorial = reasoner.execute(
    provider="openai",
    problem=f"Get started with {TOPIC}",
    goal="Deploy first infrastructure",
    constraints=["Beginner-friendly", "Use free tier"],
    model="gpt-4o-mini"
)

print("TUTORIAL:")
print(tutorial.response[:500] + "...\n")  # Preview

# ============================================================================
# STEP 3: Compile into markdown guide
# ============================================================================
markdown_guide = f"""# {TOPIC} - Beginner's Guide

## 📖 Core Concepts

{fundamentals.response}

---

## 🚀 Getting Started Tutorial

{tutorial.response}

---

## 📚 Next Steps

1. Practice the tutorial above
2. Read official documentation
3. Build a personal project
4. Join the community

---

*Generated using mycontext-ai*  
*Cost: ${fundamentals.cost_usd + tutorial.cost_usd:.4f} | Tokens: {fundamentals.tokens_used + tutorial.tokens_used}*
"""

# Save to file
with open(f"{TOPIC.lower()}_guide.md", "w", encoding="utf-8") as f:
    f.write(markdown_guide)

print(f"✅ Complete guide saved to: {TOPIC.lower()}_guide.md")
print(f"💰 Total cost: ${fundamentals.cost_usd + tutorial.cost_usd:.4f}")
print(f"📊 Total tokens: {fundamentals.tokens_used + tutorial.tokens_used}")

### Alternative: Single Context Approach (Simpler but less structured)

In [ ]:
# SIMPLER APPROACH: One context, direct markdown output

from mycontext import Context, Directive, Guidance

TOPIC = "Terraform"

# Create a comprehensive research context
research_context = Context(
    guidance=Guidance(
        role="Expert Technical Writer and Educator",
        rules=[
            "Write for complete beginners",
            "Use clear, practical examples",
            "Explain technical terms simply",
            "Include actionable steps",
            "Highlight common pitfalls"
        ],
        style="friendly, clear, practical, encouraging"
    ),
    directive=Directive(
        content=f"""Create a comprehensive beginner's guide to {TOPIC} in markdown format.

Structure:
# {TOPIC} - Beginner's Guide

## 1. What is {TOPIC}?
(2-3 paragraphs explaining what it is and why it matters)

## 2. Key Concepts & Terminology
(Explain 5-7 core concepts beginners must know)

## 3. When and Why to Use {TOPIC}
(Use cases and benefits)

## 4. Prerequisites
(What you need before starting)

## 5. Getting Started Tutorial
(Step-by-step guide with code examples)

## 6. Common Mistakes to Avoid
(5-7 common beginner mistakes)

## 7. Next Steps & Resources
(Where to learn more)

Use proper markdown formatting with headers, code blocks, bullet points, and examples.
Make it practical and immediately useful."""
    )
)

# Execute and get markdown
result = research_context.execute(
    provider="openai",
    model="gpt-4o-mini",
    temperature=0.7
)

# Save the markdown
with open(f"{TOPIC.lower()}_complete_guide.md", "w", encoding="utf-8") as f:
    f.write(result.response)

print(f"✅ Guide saved to: {TOPIC.lower()}_complete_guide.md")
print(f"💰 Cost: ${result.cost_usd:.4f}")
print(f"📊 Tokens: {result.tokens_used}")
print("\n📄 Preview:")
print(result.response[:800] + "...")

## 🎉 Next Steps

You now know the basics of mycontext-ai! Here's what to explore next:

1. **Try all 10 free cognitive templates** - Each one uses research-backed techniques
2. **Build custom templates** - Create your own `Pattern` subclasses
3. **Integrate with your favorite framework** - Works with LangChain, LlamaIndex, or standalone
4. **Optimize token usage** - Use utilities to compress and optimize contexts
5. **Track costs** - Monitor your LLM spending across providers

### Resources

- **GitHub**: https://github.com/SadhiraAI/mycontext
- **PyPI**: https://pypi.org/project/mycontext-ai/
- **Issues**: https://github.com/SadhiraAI/mycontext/issues

### Free Templates Available

1. `QuestionAnalyzer` - Deep question analysis
2. `StepByStepReasoner` - Systematic problem solving
3. `SocraticQuestioner` - Question assumptions
4. `ComparativeAnalyzer` - Compare multiple options
5. `CausalReasoner` - Analyze cause-effect
6. `RiskAssessor` - Identify and evaluate risks
7. `TradeoffAnalyzer` - Analyze competing priorities
8. And more coming soon!

Happy context engineering! 🚀